In [1]:
from __future__ import annotations

import json
import os
import platform
import random
import shutil
import sys
import tarfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content


In [2]:
%pip install -q xarray netCDF4 pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 65.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

TORNET_ARCHIVE_DIR = Path("/content/drive/MyDrive/TorNet_Backup")

expected_archives = {
    "tornet_2013.tar.gz": 2.94,
    "tornet_2014.tar.gz": 14.03,
    "tornet_2015.tar.gz": 16.20,
    "tornet_2016.tar.gz": 15.13,
    "tornet_2017.tar.gz": 14.08,
    "tornet_2018.tar.gz": 11.65,
    "tornet_2019.tar.gz": 16.97,
    "tornet_2020.tar.gz": 15.86,
    "tornet_2021.tar.gz": 17.07,
    "tornet_2022.tar.gz": 17.72,
}

rows = []

for filename, expected_size_gib in expected_archives.items():
    path = TORNET_ARCHIVE_DIR / filename
    exists = path.is_file()
    actual_size_gib = path.stat().st_size / (1024**3) if exists else None

    rows.append(
        {
            "filename": filename,
            "exists": exists,
            "expected_size_gib": expected_size_gib,
            "actual_size_gib": actual_size_gib,
            "size_difference_gib": (
                actual_size_gib - expected_size_gib
                if actual_size_gib is not None
                else None
            ),
        }
    )

archive_inventory = pd.DataFrame(rows)
archive_inventory

,filename,exists,expected_size_gib,actual_size_gib,size_difference_gib
0,tornet_2013.tar.gz,True,2.94,2.942855,0.002855
1,tornet_2014.tar.gz,True,14.03,14.031784,0.001784
2,tornet_2015.tar.gz,True,16.20,16.200937,0.000937
3,tornet_2016.tar.gz,True,15.13,15.129823,-0.000177
4,tornet_2017.tar.gz,True,14.08,14.078567,-0.001433
5,tornet_2018.tar.gz,True,11.65,11.651212,0.001212
6,tornet_2019.tar.gz,True,16.97,16.968593,-0.001407
7,tornet_2020.tar.gz,True,15.86,15.858887,-0.001113
8,tornet_2021.tar.gz,True,17.07,17.072932,0.002932
9,tornet_2022.tar.gz,True,17.72,17.717185,-0.002815


In [5]:
missing = archive_inventory.loc[~archive_inventory["exists"], "filename"].tolist()

if missing:
    raise FileNotFoundError(f"Missing TorNet archives: {missing}")

print(
    f"Found {len(archive_inventory)} archives totaling "
    f"{archive_inventory['actual_size_gib'].sum():.2f} GiB"
)

Found 10 archives totaling 141.65 GiB


In [6]:
SAMPLE_ARCHIVE = TORNET_ARCHIVE_DIR / "tornet_2013.tar.gz"
SAMPLE_OUTPUT_DIR = Path("/content/tornet_audit_samples")
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sampled_members: list[dict] = []
sampled_paths: list[Path] = []

MAX_NETCDF_SAMPLES = 6

with tarfile.open(SAMPLE_ARCHIVE, mode="r|gz") as archive:
    for member in archive:
        if not member.isfile():
            continue

        if not member.name.lower().endswith(".nc"):
            continue

        source = archive.extractfile(member)
        if source is None:
            continue

        safe_filename = member.name.replace("/", "__")
        destination = SAMPLE_OUTPUT_DIR / safe_filename

        with source, destination.open("wb") as output:
            shutil.copyfileobj(source, output)

        sampled_members.append(
            {
                "archive_member": member.name,
                "size_bytes": member.size,
                "local_sample_path": str(destination),
            }
        )
        sampled_paths.append(destination)

        if len(sampled_paths) >= MAX_NETCDF_SAMPLES:
            break

sampled_member_df = pd.DataFrame(sampled_members)
sampled_member_df

,archive_member,size_bytes,local_sample_path
0,train/2013/NUL_131101_063025_KRLX_476088s_F5.nc,1010954,/content/tornet_audit_samples/train__2013__NUL...
1,train/2013/NUL_130919_003220_KABR_477016s_D1.nc,1117203,/content/tornet_audit_samples/train__2013__NUL...
2,train/2013/NUL_130912_213227_KDIX_477860s_P9.nc,359850,/content/tornet_audit_samples/train__2013__NUL...
3,train/2013/NUL_130906_220052_KNKX_478691s_A5.nc,187443,/content/tornet_audit_samples/train__2013__NUL...
4,train/2013/NUL_131101_145852_KOKX_476238s_R4.nc,949718,/content/tornet_audit_samples/train__2013__NUL...
5,train/2013/NUL_130903_174512_KOKX_479128s_J3.nc,546816,/content/tornet_audit_samples/train__2013__NUL...


In [7]:
import xarray as xr

schema_rows = []

for sample_path in sampled_paths:
    with xr.open_dataset(sample_path, decode_times=False) as dataset:
        schema_rows.append(
            {
                "sample_path": str(sample_path),
                "dimensions": dict(dataset.sizes),
                "data_variables": sorted(dataset.data_vars),
                "coordinates": sorted(dataset.coords),
                "attributes": dict(dataset.attrs),
            }
        )

for record in schema_rows:
    print("=" * 100)
    print("FILE:", record["sample_path"])
    print("DIMENSIONS:")
    print(json.dumps(record["dimensions"], indent=2, default=str))
    print("DATA VARIABLES:")
    print(json.dumps(record["data_variables"], indent=2))
    print("COORDINATES:")
    print(json.dumps(record["coordinates"], indent=2))
    print("ATTRIBUTES:")
    print(json.dumps(record["attributes"], indent=2, default=str))

FILE: /content/tornet_audit_samples/train__2013__NUL_131101_063025_KRLX_476088s_F5.nc
DIMENSIONS:
{
  "sweep": 2,
  "time": 4,
  "lims": 2,
  "azimuth": 120,
  "range": 240
}
DATA VARIABLES:
[
  "DBZ",
  "KDP",
  "RHOHV",
  "VEL",
  "WIDTH",
  "ZDR",
  "azimuth_limits",
  "elevation",
  "frame_labels",
  "nyquist_velocity",
  "range_folded_mask",
  "range_limits"
]
COORDINATES:
[
  "azimuth",
  "range",
  "time"
]
ATTRIBUTES:
{
  "site_name": "KRLX",
  "site_lat": 38.311111,
  "site_lon": -81.723056,
  "MissingDataFlag": -999.0,
  "ef_number": -1.0,
  "event_id": "476088",
  "episode_id": "79355",
  "tornado_start_time": "",
  "tornado_end_time": "",
  "category": "NUL",
  "scit_id": "F5",
  "storm_event_url": "https://www.ncdc.noaa.gov/stormevents/eventdetails.jsp?id=476088"
}
FILE: /content/tornet_audit_samples/train__2013__NUL_130919_003220_KABR_477016s_D1.nc
DIMENSIONS:
{
  "sweep": 2,
  "time": 4,
  "lims": 2,
  "azimuth": 120,
  "range": 240
}
DATA VARIABLES:
[
  "DBZ",
  "KDP",


In [8]:
variable_rows = []

for sample_path in sampled_paths:
    with xr.open_dataset(sample_path, decode_times=False) as dataset:
        for variable_name, variable in dataset.variables.items():
            values = np.asarray(variable.values)

            numeric_values = (
                values.astype(np.float64, copy=False)
                if np.issubdtype(values.dtype, np.number)
                else None
            )

            finite_fraction = None
            minimum = None
            maximum = None

            if numeric_values is not None and numeric_values.size:
                finite_mask = np.isfinite(numeric_values)
                finite_fraction = float(finite_mask.mean())

                if finite_mask.any():
                    minimum = float(numeric_values[finite_mask].min())
                    maximum = float(numeric_values[finite_mask].max())

            variable_rows.append(
                {
                    "sample": sample_path.name,
                    "variable": variable_name,
                    "dimensions": tuple(variable.dims),
                    "shape": tuple(variable.shape),
                    "dtype": str(variable.dtype),
                    "finite_fraction": finite_fraction,
                    "minimum": minimum,
                    "maximum": maximum,
                    "attributes": dict(variable.attrs),
                }
            )

variable_inventory = pd.DataFrame(variable_rows)
variable_inventory

,sample,variable,dimensions,shape,dtype,finite_fraction,minimum,maximum,attributes
0,train__2013__NUL_131101_063025_KRLX_476088s_F5.nc,elevation,"(sweep,)","(2,)",float64,1.000000,5.000000e-01,9.000000e-01,"{'units': 'degrees', 'long_name': 'elevation_a..."
1,train__2013__NUL_131101_063025_KRLX_476088s_F5.nc,frame_labels,"(time,)","(4,)",uint8,1.000000,0.000000e+00,0.000000e+00,"{'units': 'binary', 'description': 'Value of 1..."
2,train__2013__NUL_131101_063025_KRLX_476088s_F5.nc,nyquist_velocity,"(time, sweep)","(4, 2)",float32,1.000000,2.837000e+01,2.837000e+01,"{'units': 'm/s', 'long_name': 'nyquist_velocity'}"
3,train__2013__NUL_131101_063025_KRLX_476088s_F5.nc,azimuth_limits,"(lims,)","(2,)",float32,1.000000,2.790000e+02,3.390000e+02,"{'units': 'degrees', 'long_name': 'az_limits_o..."
4,train__2013__NUL_131101_063025_KRLX_476088s_F5.nc,range_limits,"(lims,)","(2,)",float32,1.000000,3.852400e+04,9.852400e+04,"{'units': 'meters', 'long_name': 'range_limits..."
...,...,...,...,...,...,...,...,...,...
85,train__2013__NUL_130903_174512_KOKX_479128s_J3.nc,WIDTH,"(time, azimuth, range, sweep)","(4, 120, 240, 2)",float32,0.253524,0.000000e+00,1.550000e+01,"{'units': 'm/s', 'standard_name': 'doppler_spe..."
86,train__2013__NUL_130903_174512_KOKX_479128s_J3.nc,range_folded_mask,"(time, azimuth, range, sweep)","(4, 120, 240, 2)",uint8,1.000000,0.000000e+00,1.000000e+00,"{'units': 'binary', 'description': 'Field is 1..."
87,train__2013__NUL_130903_174512_KOKX_479128s_J3.nc,azimuth,"(azimuth,)","(120,)",float32,1.000000,3.325000e+01,9.275000e+01,"{'units': 'degrees', 'long_name': 'azimuth_ang..."
88,train__2013__NUL_130903_174512_KOKX_479128s_J3.nc,range,"(range,)","(240,)",float32,1.000000,1.368050e+05,1.965550e+05,"{'units': 'meters', 'long_name': 'range_from_i..."
